In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import math
import numpy as np
import matplotlib.pyplot as plt
from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP
from micrograd.utils import draw_dot

%matplotlib inline

In [ ]:
def f(x):
    return 3*x**2 - 4*x + 5

f(3.0)

xs = np.arange(-5, 5, 0.25)
ys = f(xs)
plt.plot(ys)

In [ ]:
# example 1
# pick a small 'h'

h = 0.0000001
x = 3

# then we push the function by h, and check how function is responded to that push and normilize by 'h' to get a slope
(f(x + h) - f(x)) / h

In [ ]:
# example 2 (more inputs)
h = 0.0001

# inputs
a = 2.0
b = -3.0
c = 10.0

d1 = a * b + c
a += h # d with respect to 'a'
d2 = a * b + c

print(f"d1: {d1}")
print(f"d2: {d2}")
print(f"slope: {(d2 - d1) / h}")

In [ ]:
# example 2.1 (with using Value class from engine)
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')

e = a * b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value(-2.0, label='f')

# let's create an output for our graph (forward pass)
L = d * f; L.label='L'; L.grad=1.0

In [ ]:
# so what we want to know about L
# dL/dd = ?
# Definition of derivative is:
# (f(x+h) - f(x)) / h ->
# ((d+h)*f - d*f)) / h ->
# (d*f + h*f - d*f) / h ->
# (h*f) / h ->
# f
f.grad = 4.0
d.grad = -2.0

c.grad = -2.0
e.grad = -2.0

a.grad = (-2.0 * -3.0)
b.grad = (-2.0 * 2.0)

In [ ]:
# apply single optimization step
# we want to nudge out inputs, to make L go up

a.data += 0.01 * a.grad
b.data += 0.01 * b.grad
c.data += 0.01 * c.grad
f.data += 0.01 * f.grad

e = a * b
d = e + c
L = d * f

print(L.data)

In [ ]:
draw_dot(L)

In [ ]:
# example 2.2 (derivative of L with respect to L)

def example(h=0.0001):
    a = Value(2.0, label='a')
    b = Value(-3.0, label='b')
    c = Value(10.0, label='c')

    e = a * b; e.label = 'e'
    d = e + c; d.label = 'd'
    f = Value(-2.0, label='f')

    L = d * f; L.label='L'
    L1 = L.data

    # place h to Value definition to get a gradient value
    a = Value(2.0, label='a')
    b = Value(-3.0, label='b')
    c = Value(10.0, label='c')

    e = a * b; e.label = 'e'
    e.data +=h
    d = e + c; d.label = 'd'
    f = Value(-2.0, label='f')

    L = d * f; L.label='L'
    L2 = L.data

    print((L2 - L1) / h)

example()


# backpropagation explanation

For example we need to derive `dL/dc` (derivative of `L` with respect to `c`)\
How `L` is sensitive to `c` through `d`, or how `c` impacts `L` through `d` ?

Let's take a look at the following steps
1. What is `dd / dc` ?
    - So `d = c + e`, which means that `dd / dc` is `1`
        1. `(f(x+h) - f(x)) / h`
        2. `(((c+h) + e) - (c + e))) / h`
        3. `h/h -> 1`
2. What is `dd / de` ?
    - So as we know, that `d = c + e`, then `dd / de` is also `1`
3. Apply **Chain Rule**, to derive `dL/dc`
    - DO:
        - `dL / dc = (dL / dd) * (dd / dc)`
    - HAVE (information about local derivatives):
        - How `d` impacts `L`, `-2.0`
        - How `c` impacts `d`, `1.0`
        - How `e` impacts `d`, `1.0`

Next, we now know that `dL / de = -2.0`, what is `dL / da` ?
- So the Chain Rule tells us, that `dL / da = (dL / de) * (de / da)`
- Next we have to find the `de / da`, which is `-3.0`
- Then `dL / da = -2.0 * -3.0`




In [ ]:
# example 3 (backpropagate through a Neuron)
# activation function, like sigmoid or tanh, serves as some kind of "squashing" function on extremes, check the graph below

plt.plot(np.arange(-5, 5, 0.2), np.tanh(np.arange(-5, 5, 0.2))); plt.grid();

In [ ]:
# 2D Neuron
# inputs x1, x2 with corresponding weights w1, w2 (Synaptic strength for each input)
x1 = Value(2.0, label='x1'); w1 = Value(-3.0, label='w1')
x2 = Value(0.0, label='x2'); w2 = Value(1.0, label='w2')

# bias
b = Value(6.75, label='b')

x1w1 = x1*w1; x1w1.label = 'x1w1'
x2w2 = x2*w2; x2w2.label = 'x2w2'

x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1w1x2w2'

N = x1w1x2w2 + b; N.label = 'N'

# output
o = N.tanh(); o.label = 'o'

In [ ]:
# let's backpropagate manually again

# Derivate of 'o' with respect to 'o', base case
o.grad = 1.0

# What is do/dN?
# do/dN = 1 - tanh(N)**2
# do/dN = 1 - o**2
N.grad = 1 - o.data**2

# As we know from previous example, The "+" op node is just a distributor of gradient, so...
x1w1x2w2.grad = N.grad
b.grad = N.grad

# same goes for the following ones
x1w1.grad = x1w1x2w2.grad
x2w2.grad = x1w1x2w2.grad

# apply chain rule on "*" op nodes
x1.grad = w1.data * x1w1.grad; w1.grad = x1.data * x1w1.grad
x2.grad = w2.data * x2w2.grad; w2.grad = x2.data * x2w2.grad

In [ ]:
draw_dot(o)

In [ ]:
# example 3.1 (implementing backward function and automatic gradient)
# inputs and weights

# 2D Neuron
# inputs x1, x2 with corresponding weights w1, w2 (Synaptic strength for each input)
x1 = Value(2.0, label='x1'); w1 = Value(-3.0, label='w1')
x2 = Value(0.0, label='x2'); w2 = Value(1.0, label='w2')

# bias
b = Value(6.8813735870195432, label='b')

x1w1 = x1*w1; x1w1.label = 'x1w1'
x2w2 = x2*w2; x2w2.label = 'x2w2'

x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1w1x2w2'

N = x1w1x2w2 + b; N.label = 'N'

# output
o = N.tanh(); o.label = 'o'
o.backward()

In [ ]:
draw_dot(o)

In [ ]:
# example 3.2 (let's divide tanh into atomic parts)
# inputs x1,x2
x1 = Value(2.0, label='x1'); w1 = Value(-3.0, label='w1')
x2 = Value(0.0, label='x2'); w2 = Value(1.0, label='w2')


# bias
b = Value(6.75, label='b')

x1w1 = x1*w1; x1w1.label = 'x1w1'
x2w2 = x2*w2; x2w2.label = 'x2w2'

x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1w1x2w2'

N = x1w1x2w2 + b; N.label = 'N'

# ----
e = (2*N).exp()
o = (e - 1) / (e + 1)
# ----
o.label = 'o'

o.backward()

In [ ]:
draw_dot(o)

In [ ]:
# example 3.3 (torch equivalent)
import torch

x1 = torch.Tensor([2.0]).double()   ; x1.requires_grad = True
x2 = torch.Tensor([0.0]).double()   ; x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double()  ; w1.requires_grad = True
w2 = torch.Tensor([1.0]).double()   ; w2.requires_grad = True
b = torch.Tensor([6.75]).double()   ; b.requires_grad = True

n = x1 * w1 + x2 * w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()

print("--------------")
print(f'x2', x2.grad.item())
print(f'w2', w2.grad.item())
print(f'x1', x1.grad.item())
print(f'w1', w1.grad.item())

In [ ]:
# example 4.1 (building neural net library (MLP))

x = [2.0, 3.0, -1]
n = MLP(len(x), [4, 4, 1])
n(x)

In [ ]:
# example 4.2 MLP with Dataset, implementing Loss function and Gradient Descent

xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

In [ ]:
# step 1: Forward pass
ypred = [n(x) for x in xs]
loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])
loss

In [ ]:
# step 2: Backward pass
loss.backward()

In [ ]:
# step 3: nudge the gradient vector by tiny amount to minimize the loss
learning_rate = 0.05
for p in n.parameters():
    p.data += -learning_rate * p.grad

print(ypred)
# step 4: automate step 1-3

In [ ]:
# example 4.3 Training loop

x = [2.0, 3.0, -1]
n = MLP(len(x), [8, 8, 1])

xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]
ys = [1.0, -1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0]

learning_rate = 0.05
epochs = 300

for epoch in range(epochs):
    # forward
    ypred = [n(x) for x in xs]
    loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])

    # backward
    for p in n.parameters():
        p.grad = 0.0 # very important, reset your gradient vector, before backward pass
    loss.backward()

    # update weights
    for p in n.parameters():
        p.data += -learning_rate * p.grad
    
    if epoch % 10 == 0:
        print(f"epocs: {epoch}, loss: {loss.data}")

print(f"predictions: {[pr.data for pr in ypred]}")

In [ ]:
draw_dot(loss)